In [36]:
import numpy as np
from time import perf_counter

### Detector class

In [37]:
class Detector():
    def __init__(self, R: float, N: int, r: float, a: float):
        """
        Parameters
        ----------
        R : float
            Радиус детектора в метрах.
        N : int
            Количество фотоумножителей.
        r : float
            Радиус фотокатода в метрах.
        a : float
            Эффективность фотокатода.
        """
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        self.rng = np.random.default_rng()

    def coord_cart(self, theta: float, phi: float) -> np.ndarray:
        return np.array([self.R * np.sin(theta)*np.cos(phi), self.R * np.sin(theta)*np.sin(phi), self.R*np.cos(theta)])

    def fill_detector(self):
        self.photo_multipliers = np.zeros((self.N, 3))
        S = 4 * self.R**2 * np.pi 
        delta_S = S / self.N 
        lin_size = delta_S**0.5 
        N_theta = np.pi * self.R / lin_size 
        delta_theta = np.pi / N_theta

        theta_floor = [int(np.floor(N_theta)), np.pi / np.floor(N_theta), delta_theta - np.pi / np.floor(N_theta)]
        theta_ceil = [int(np.ceil(N_theta)), np.pi / np.ceil(N_theta), delta_theta - np.pi / np.ceil(N_theta)]

        best_theta = min(theta_floor, theta_ceil, key=lambda x: abs(x[2]))
        N_theta, delta_theta = best_theta[0], best_theta[1]

        mid_thetas = np.array([delta_theta / 2 + delta_theta*i for i in range(0, N_theta)])

        n_phi = np.array([2, 7, 11, 11, 7, 2])     

        phi_phases = np.zeros(N_theta)
        for i in range(N_theta):
            if i == 0:
                continue
            phi_phases[i] = np.pi / np.lcm(int(n_phi[i-1]), int(n_phi[i]))

        n = 0
        phase = 0
        for i in range(N_theta):
            if i < 3:
                phase += phi_phases[i]
            else:
                phase -= phi_phases[i]
            for j in range(int(n_phi[i])):
                self.photo_multipliers[n] = self.coord_cart(mid_thetas[i], 2*np.pi/n_phi[i] * j + phase)
                n += 1

    def in_chord(self, coord: np.ndarray) -> int:
        for k, photo_multiplier in enumerate(self.photo_multipliers):
            distance = np.linalg.norm(photo_multiplier-coord)
            if distance <= self.r:
                return k
        return -1

    def photo_electrons(self) -> int:
        return self.rng.poisson(lam=self.a)

    def register(self, coord: np.ndarray) -> int:
        id = self.in_chord(coord)
        if id < 0:
            return 0
        return self.photo_electrons()

    def register_many(self, hits: np.ndarray) -> int:
        diff = hits[:, None, :] - self.photo_multipliers[None, :, :]
        distances2 = np.sum(diff**2, axis=2)

        hit_mask = distances2 <= self.r**2

        photons_per_pmt = np.sum(hit_mask, axis=0)

        charge_per_pmt = self.rng.poisson(
            lam=self.a * photons_per_pmt
        )

        return np.sum(charge_per_pmt)
            

### Source class

In [38]:
class Source:
    def __init__(self, pos: np.ndarray):
        """
        Parameters
        ----------
        pos : np.array
            координаты источника
        """
        self.Y = 10000 
        self.pos = pos

    def sphere(self, R: float) -> np.ndarray: #взял метод из ROOT
        r2 = 1.0

        while r2 > 0.25:
            a = np.random.random() - 0.5
            b = np.random.random() - 0.5
            r2 = a**2 + b**2

        z = R * (-1.0 + 8.0 * r2)
        scale = 8.0 * R * np.sqrt(0.25 - r2)

        x = a * scale
        y = b * scale

        point = np.array([x, y, z])

        return point

    def generate_photons(self, E: float) -> np.ndarray:
        photon_number = int(E * self.Y)
        photons = np.zeros((photon_number, 3))
        for i in range(photon_number):
            photons[i] = self.sphere(1.0)
        return photons

    def generate_photons_vec(self, E: float) -> np.ndarray:
        photon_number = int(E * self.Y)

        phi = np.random.uniform(0, 2*np.pi, photon_number)
        cos_theta = np.random.uniform(-1, 1, photon_number)
        sin_theta = np.sqrt(1 - cos_theta**2)

        photons = np.column_stack((
            sin_theta * np.cos(phi),
            sin_theta * np.sin(phi),
            cos_theta
        ))

        return photons


In [39]:
def photon_propagation(pos: np.ndarray, direction: np.ndarray, R: float) -> np.ndarray:
    nr = np.dot(pos, direction)
    r = np.linalg.norm(pos)

    t = -nr + (nr**2 - (r**2 - R**2))**0.5
    hit = pos + direction * t
    return hit

def photon_propagation_vec(pos: np.ndarray, directions: np.ndarray, R: float) -> np.ndarray:
    nr = directions @ pos
    r2 = np.dot(pos, pos)

    t = -nr + np.sqrt(nr**2 - (r2 - R**2))

    hits = pos + directions * t[:, None]

    return hits

In [40]:
R = 2
N = 40
r = 0.1
a = 0.5

detector = Detector(R, N, r, a)
detector.fill_detector()
source_pos = np.array([0.5, 0.5, 0.5])
source = Source(source_pos)

In [41]:
N_event = 1000
E = 0.2

charges_old = np.zeros(N_event)
start = perf_counter()

for event in range(N_event):
    photons = source.generate_photons(E)
    for photon in photons:
        hit = photon_propagation(source_pos, photon, detector.R)
        charges_old[event] += detector.register(hit)

time_old = perf_counter() - start

charges_vec = np.zeros(N_event)
start = perf_counter()

for event in range(N_event):
    photons = source.generate_photons_vec(E)
    hits = photon_propagation_vec(source_pos, photons, detector.R)
    charges_vec[event] = detector.register_many(hits)

time_vec = perf_counter() - start

print(f'Обычный питон: {time_old} с, на одно событие {time_old/N_event}')
print(f'Векторизация: {time_vec} с, на одно событие {time_vec/N_event}')
print(f'Ускорение: {time_old / time_vec} раз')

Обычный питон: 59.48016050015576 с, на одно событие 0.05948016050015576
Векторизация: 1.0351956670638174 с, на одно событие 0.0010351956670638174
Ускорение: 57.457891674588076 раз
